# SQLite 与事务

学习目标：用 sqlite3 保存和查询小型记录，正确绑定 SQL 参数，并通过提交、回滚与资源关闭控制数据何时保留。

前置知识：列表与元组、字典、函数和类型标注、异常处理、with、contextlib.closing、临时目录与路径操作。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用内存数据库或自动清理的临时目录，无需独立数据库服务。

## 1 连接、游标与关闭责任

SQLite 是可以嵌入程序的数据库，不需要独立服务器进程；sqlite3 是 Python 提供的接口。连接（connection）代表与数据库的会话，游标（cursor）负责执行 SQL 并读取结果。

sqlite3.connect(":memory:", autocommit=False) 创建内存数据库连接。autocommit 是 Python 3.12 新增的事务控制参数；本章主要显式使用 False，并在需要保留修改时提交，在需要放弃修改时回滚。

连接的 with 管理事务，不负责关闭连接；这与文件的 with 不同。下面用外层 closing 关闭连接，内层 closing 先关闭游标，避免依赖垃圾回收。

In [1]:
import contextlib
import sqlite3

with contextlib.closing(
    sqlite3.connect(":memory:", autocommit=False)
) as connection:
    with contextlib.closing(connection.cursor()) as cursor:
        cursor.execute("SELECT 2 + 3")
        print(cursor.fetchone())  # (5,)：一行结果默认是元组。

# 退出顺序为游标、连接；这里没有要保留的数据库修改。
try:
    connection.cursor()
except sqlite3.ProgrammingError as error:
    print(type(error).__name__)  # ProgrammingError：已关闭的连接不能继续使用。
else:
    raise AssertionError("关闭后的连接应拒绝新建游标")

(5,)
ProgrammingError


## 2 建表、写入与查询

SQL 是操作数据库的语言。表按列定义字段，每行保存一条记录；本章的 studies 表保存 course（课程名）和 minutes（学习分钟数）。

| SQL 写法 | 中文名称／含义 |
| --- | --- |
| CREATE TABLE | 创建表并声明列与约束 |
| INSERT INTO | 插入记录 |
| SELECT | 选择需要读取的列或表达式 |
| FROM | 指定读取的表 |
| WHERE | 按条件筛选行 |
| ORDER BY | 按指定列或表达式排列结果 |

course 使用 NOT NULL 和 UNIQUE，要求不为空值且不重复；minutes 使用 NOT NULL 和 CHECK(minutes >= 0)，要求不为空值并满足非负检查。示例输入均使用 Python 整数分钟数，SQL 约束不代替应用的完整输入校验。

execute 执行一条语句；executemany 针对每组参数重复执行同一条写入语句。后续反复使用同一个小表，因此用函数封装建表和初始写入；函数关闭自己的游标，提交仍由调用者决定。

In [2]:
def create_studies(connection: sqlite3.Connection) -> None:
    """在新数据库中建立示例表和初始记录，不提交调用者的事务。"""
    with contextlib.closing(connection.cursor()) as cursor:
        # 1. 课程名称唯一；分钟数不允许为空或小于零。
        cursor.execute(
            "CREATE TABLE studies ("
            "course TEXT NOT NULL UNIQUE, "
            "minutes INTEGER NOT NULL CHECK(minutes >= 0))"
        )
        # 2. 占位符对应每条记录的课程名和分钟数，不拼接 Python 字符串。
        cursor.executemany(
            "INSERT INTO studies(course, minutes) VALUES (?, ?)",
            [("python", 30), ("sql", 20), ("web", 15)],
        )


with contextlib.closing(
    sqlite3.connect(":memory:", autocommit=False)
) as connection:
    create_studies(connection)
    connection.commit()  # 提交初始数据，后续回滚不会撤销这次提交。
    with contextlib.closing(connection.cursor()) as cursor:
        cursor.execute("SELECT course, minutes FROM studies ORDER BY course")
        print(cursor.fetchall())
        # [('python', 30), ('sql', 20), ('web', 15)]

[('python', 30), ('sql', 20), ('web', 15)]


## 3 参数绑定传递值

### 3.1 问号与命名占位符

SQL 文本和参数分开传给 execute；占位符表示值的位置，不要用 f-string 或字符串拼接把输入放进 SQL，也不要给问号再加引号。

| 占位写法 | 中文名称／含义 | 第二个参数 |
| --- | --- | --- |
| ? | 按位置绑定一个值 | 长度与占位符数量一致的序列 |
| :course | 按名称绑定一个值，course 是参数名 | 包含 course 键的字典 |

只有一个位置参数时仍需传入单元素元组，例如 ("python",)，逗号不能省略。命名参数字典中的键不包含冒号。

下面把带单引号和 SQL 片段的输入当作普通课程名查询；绑定机制不会把其中内容重新解释为查询条件。

In [3]:
with contextlib.closing(
    sqlite3.connect(":memory:", autocommit=False)
) as connection:
    create_studies(connection)
    connection.commit()
    with contextlib.closing(connection.cursor()) as cursor:
        cursor.execute(
            "SELECT minutes FROM studies WHERE course = ?", ("python",)
        )
        print(cursor.fetchone())  # (30,)

        cursor.execute(
            "SELECT course FROM studies "
            "WHERE minutes >= :minimum ORDER BY course",
            {"minimum": 20},
        )
        print(cursor.fetchall())  # [('python',), ('sql',)]

        cursor.execute(
            "SELECT course FROM studies WHERE course = ?",
            ("sql' OR 1=1 --",),
        )
        print(cursor.fetchall())  # []：数据库中没有这个完整的课程名。

(30,)
[('python',), ('sql',)]
[]


### 3.2 参数数量必须与占位符匹配

使用问号时，参数序列长度必须匹配；命名绑定时，字典必须包含所有用到的参数名。缺失参数会引发 sqlite3.ProgrammingError，不能把这个异常当作“查询没有结果”。

下面只执行常量表达式，不创建表，也不产生数据库修改。

In [4]:
with contextlib.closing(
    sqlite3.connect(":memory:", autocommit=False)
) as connection:
    with contextlib.closing(connection.cursor()) as cursor:
        for sql, parameters in (
            ("SELECT ?, ?", (1,)),
            ("SELECT :course", {}),
        ):
            try:
                cursor.execute(sql, parameters)
            except sqlite3.ProgrammingError as error:
                print(type(error).__name__)  # 两次都应为 ProgrammingError。
            else:
                raise AssertionError("缺少 SQL 绑定参数应失败")

ProgrammingError
ProgrammingError


## 4 参数不能替换表名和列名

参数是表达式中的值，不能替换表名、列名或排序方向等 SQL 结构。例如 SELECT ? 会选择绑定的常量，不会把字符串 "minutes" 解释为 minutes 列。

需要选择排序方式时，可以让应用中的固定选项对应完整 SQL，输入只负责选择选项。下面的 course 和 minutes 是应用允许的两种排序方式，SQL 文本均由程序预先定义。

In [5]:
queries_by_order = {
    "course": "SELECT course, minutes FROM studies ORDER BY course",
    "minutes": (
        "SELECT course, minutes FROM studies ORDER BY minutes, course"
    ),
}
with contextlib.closing(
    sqlite3.connect(":memory:", autocommit=False)
) as connection:
    create_studies(connection)
    connection.commit()
    with contextlib.closing(connection.cursor()) as cursor:
        cursor.execute(
            "SELECT ? FROM studies ORDER BY course", ("minutes",)
        )
        print(cursor.fetchall())  # [('minutes',), ('minutes',), ('minutes',)]

        cursor.execute(queries_by_order["minutes"])
        print(cursor.fetchall())  # [('web', 15), ('sql', 20), ('python', 30)]

[('minutes',), ('minutes',), ('minutes',)]
[('web', 15), ('sql', 20), ('python', 30)]


## 5 逐步读取结果

读取方法会推进当前结果集，不会每次从第一行重读。fetchall 读取的是剩余行；结果很大时，一次读取全部行也会形成较大的 Python 列表。

| API／属性 | 中文名称／含义 |
| --- | --- |
| fetchone() | 读取下一行，没有剩余行时返回 None |
| fetchmany(size) | 最多读取 size 行，size 是本次期望的行数 |
| fetchall() | 读取全部剩余行，返回列表 |
| rowcount | 对本章普通写入语句记录影响行数；对 SELECT 为 -1 |

多行查询需要确定顺序时必须写 ORDER BY；相同排序值还需要附加区分字段，不能依赖插入顺序。游标也支持迭代，后面的按列名读取示例会直接遍历它。

In [6]:
with contextlib.closing(
    sqlite3.connect(":memory:", autocommit=False)
) as connection:
    create_studies(connection)
    connection.commit()
    with contextlib.closing(connection.cursor()) as cursor:
        cursor.execute("SELECT course, minutes FROM studies ORDER BY course")
        print(cursor.rowcount)  # -1：不能用它判断 SELECT 返回几行。
        print(cursor.fetchone())  # ('python', 30)
        print(cursor.fetchmany(1))  # [('sql', 20)]
        print(cursor.fetchall())  # [('web', 15)]：只剩这一行。
        print(cursor.fetchone(), cursor.fetchall())  # None []：均已耗尽。

    try:
        cursor.fetchone()
    except sqlite3.ProgrammingError as error:
        print(type(error).__name__)  # ProgrammingError：游标已经关闭。
    else:
        raise AssertionError("已关闭的游标应拒绝读取")

-1
('python', 30)
[('sql', 20)]
[('web', 15)]
None []
ProgrammingError


## 6 用 Row 按列名读取

默认结果行是 tuple，按查询列的顺序读取。将 connection.row\_factory 设为 sqlite3.Row 后，新创建游标的结果行同时支持位置索引和不区分大小写的列名访问；keys() 可查看列名。

应在创建读取游标之前设置 row\_factory；它不会追溯修改已存在游标的设置。查询使用明确列名，可以让结果与业务字段的对应关系更清楚。

In [7]:
with contextlib.closing(
    sqlite3.connect(":memory:", autocommit=False)
) as connection:
    create_studies(connection)
    connection.commit()
    connection.row_factory = sqlite3.Row
    with contextlib.closing(connection.cursor()) as cursor:
        cursor.execute("SELECT course, minutes FROM studies ORDER BY course")
        for row in cursor:
            print(row["course"], row["MINUTES"], row[0])
        # python 30 python；sql 20 sql；web 15 web。
        print(row.keys())  # ['course', 'minutes']

python 30 python
sql 20 sql
web 15 web
['course', 'minutes']


## 7 显式提交和回滚

### 7.1 autocommit=False 保持事务打开

事务（transaction）把需要共同保留或撤销的操作放在同一边界内。commit 提交当前事务，rollback 撤销当前事务中的未提交修改，不能撤销更早已经完成的提交。

在 Python 3.12 的 autocommit=False 模式下，connect 会隐式开启事务，commit 或 rollback 完成后也会立即开启新事务；底层使用 BEGIN DEFERRED。它先建立事务边界，实际读写锁随数据库访问取得，不代表连接一创建就锁住整张表。

in\_transaction 反映底层事务是否处于活动状态，不能当作“已经修改了数据”的标志。下面没有写入，仍能观察到 True。

In [8]:
with contextlib.closing(
    sqlite3.connect(":memory:", autocommit=False)
) as connection:
    print(connection.in_transaction)  # True：已进入事务，但还没有数据修改。
    connection.commit()
    print(connection.in_transaction)  # True：提交后立即开启下一事务。
    connection.rollback()
    print(connection.in_transaction)  # True：回滚后也立即开启下一事务。

True
True
True


### 7.2 修改成功不等于已经提交

UPDATE 用 SET 指定修改内容，用 WHERE 限定记录；DELETE FROM 删除符合 WHERE 的记录。省略 WHERE 会作用于整张表，因此下面始终明确目标课程。

同一连接能查询自己的未提交修改；这不证明修改已经持久保存。沿用初始表，本例先修改再回滚，接着重新修改并提交，最后删除另一条记录再回滚。

In [9]:
with contextlib.closing(
    sqlite3.connect(":memory:", autocommit=False)
) as connection:
    create_studies(connection)
    connection.commit()
    with contextlib.closing(connection.cursor()) as cursor:
        cursor.execute(
            "UPDATE studies SET minutes = ? WHERE course = ?",
            (45, "python"),
        )
        print(cursor.rowcount)  # 1：执行影响一行，但还没有提交。
        cursor.execute(
            "SELECT minutes FROM studies WHERE course = ?", ("python",)
        )
        print(cursor.fetchone())  # (45,)：自己能看见尚未提交的值。
        connection.rollback()

        cursor.execute(
            "SELECT minutes FROM studies WHERE course = ?", ("python",)
        )
        print(cursor.fetchone())  # (30,)：回到上次提交的数据。

        cursor.execute(
            "UPDATE studies SET minutes = ? WHERE course = ?",
            (60, "python"),
        )
        connection.commit()
        cursor.execute("DELETE FROM studies WHERE course = ?", ("web",))
        connection.rollback()
        cursor.execute("SELECT course, minutes FROM studies ORDER BY course")
        print(cursor.fetchall())
        # [('python', 60), ('sql', 20), ('web', 15)]：只撤销删除。

1
(45,)
(30,)
[('python', 60), ('sql', 20), ('web', 15)]


### 7.3 没有匹配行并不自动报错

UPDATE 或 DELETE 的条件没有匹配任何行时，语句仍可成功，影响行数为 0。如果业务要求目标必须存在，就需要检查 rowcount 并决定如何处理。

这里仅观察 SQL 行为，不把“语句执行完成”当作“目标记录已经更新”。

In [10]:
with contextlib.closing(
    sqlite3.connect(":memory:", autocommit=False)
) as connection:
    create_studies(connection)
    connection.commit()
    with contextlib.closing(connection.cursor()) as cursor:
        cursor.execute(
            "UPDATE studies SET minutes = ? WHERE course = ?",
            (50, "missing"),
        )
        print(cursor.rowcount)  # 0：目标不存在，数据库没有替应用报错。
        assert cursor.rowcount == 0
    connection.rollback()

0


## 8 with 管理事务，不管理连接寿命

### 8.1 正常退出提交，外层负责关闭

在本章 autocommit=False 模式下，with connection 正常退出会提交事务；代码块有未捕获异常，或提交本身失败时，会尝试回滚，并让异常继续传播。

事务边界与资源边界分开：外层 closing 管连接寿命，with connection 管这段操作的提交或回滚，内部 closing 管游标。正常离开 with connection 后，连接仍可继续使用。

In [11]:
with contextlib.closing(
    sqlite3.connect(":memory:", autocommit=False)
) as connection:
    with connection:
        create_studies(connection)  # 正常退出，提交建表和初始记录。

    with connection:
        with contextlib.closing(connection.cursor()) as cursor:
            cursor.execute(
                "UPDATE studies SET minutes = ? WHERE course = ?",
                (45, "python"),
            )

    # 事务上下文已经退出，但连接仍未关闭。
    with contextlib.closing(connection.cursor()) as cursor:
        cursor.execute(
            "SELECT minutes FROM studies WHERE course = ?", ("python",)
        )
        print(cursor.fetchone())  # (45,)：第二个事务的修改已提交。

(45,)


### 8.2 异常应越过事务上下文边界

executemany 会逐组执行参数；中间某条失败，不代表前面的写入已经被整个批次自动撤销。要让一批操作共同成功或失败，把它们放在同一个事务中。

下面先插入新的 rust，再插入已经存在的 python，触发 UNIQUE 约束错误 sqlite3.IntegrityError。异常在 with connection 外捕获，事务上下文才能看到失败并回滚整批操作。

如果在 with connection 内捕获异常后正常走到出口，管理器会按正常退出处理；不要这样意外提交前半批数据。

In [12]:
with contextlib.closing(
    sqlite3.connect(":memory:", autocommit=False)
) as connection:
    create_studies(connection)
    connection.commit()
    try:
        with connection:
            with contextlib.closing(connection.cursor()) as cursor:
                cursor.executemany(
                    "INSERT INTO studies(course, minutes) VALUES (?, ?)",
                    [("rust", 10), ("python", 99)],
                )
    except sqlite3.IntegrityError as error:
        print(type(error).__name__)  # IntegrityError：python 违反唯一约束。
    else:
        raise AssertionError("重复课程名应使整批写入失败")

    with contextlib.closing(connection.cursor()) as cursor:
        cursor.execute("SELECT course, minutes FROM studies ORDER BY course")
        after_failure = cursor.fetchall()
        print(after_failure)
        # [('python', 30), ('sql', 20), ('web', 15)]：rust 也被回滚。
        assert after_failure == [("python", 30), ("sql", 20), ("web", 15)]

IntegrityError
[('python', 30), ('sql', 20), ('web', 15)]


## 9 区分 Python 3.12 的三种事务模式

### 9.1 省略 autocommit 仍是旧模式

Python 3.12 虽然新增 autocommit，但省略它时的默认值仍为 sqlite3.LEGACY\_TRANSACTION\_CONTROL。新参数存在，不代表默认行为已经改为 False。

| autocommit 值 | 中文名称／含义 | 本章关注的行为 |
| --- | --- | --- |
| False | 按 Python DB-API 约定管理事务 | 连接后保持事务打开，显式提交或回滚 |
| True | 使用 SQLite 的自动提交模式 | Python commit 和 rollback 不起作用 |
| sqlite3.LEGACY\_TRANSACTION\_CONTROL | Python 3.12 之前的事务控制方式 | 是否隐式开启事务由 isolation\_level 决定 |

只有旧模式会使用 isolation\_level。其默认不是 None：INSERT、UPDATE、DELETE、REPLACE 在没有事务时会隐式开启事务；CREATE TABLE 和 SELECT 不会因此由 Python 隐式开启事务。这里专门省略 autocommit，观察默认行为。

In [13]:
with contextlib.closing(sqlite3.connect(":memory:")) as legacy_connection:
    print(
        legacy_connection.autocommit == sqlite3.LEGACY_TRANSACTION_CONTROL
    )  # True：本单元专门观察 Python 3.12 的默认值。
    print(legacy_connection.in_transaction)  # False
    with contextlib.closing(legacy_connection.cursor()) as cursor:
        cursor.execute("CREATE TABLE notes(content TEXT)")
        print(legacy_connection.in_transaction)  # False
        cursor.execute("INSERT INTO notes(content) VALUES (?)", ("复习",))
        print(legacy_connection.in_transaction)  # True：写入前隐式开启事务。
        legacy_connection.rollback()
        cursor.execute("SELECT content FROM notes")
        print(cursor.fetchall())  # []：回滚插入，表仍存在。

True
False
False
True
[]


### 9.2 自动提交模式不能靠 rollback 撤销已完成语句

autocommit=True 使用 SQLite 的自动提交模式；本例不手写 BEGIN，每条语句完成后，其自动事务也完成。Python 的 commit、rollback 不起作用，with connection 也不会替一批语句创建共同的事务。

需要前后两步共同回滚时，应选定合适的事务模式和边界，不能只增加一个 with 或在末尾调用 rollback。

In [14]:
with contextlib.closing(
    sqlite3.connect(":memory:", autocommit=True)
) as connection:
    with connection:
        with contextlib.closing(connection.cursor()) as cursor:
            cursor.execute("CREATE TABLE notes(content TEXT)")
            cursor.execute(
                "INSERT INTO notes(content) VALUES (?)", ("已经完成",)
            )
    connection.rollback()  # 此模式下不做回滚。
    with contextlib.closing(connection.cursor()) as cursor:
        cursor.execute("SELECT content FROM notes")
        print(cursor.fetchall())  # [('已经完成',)]：先前语句已经提交。
    print(connection.in_transaction)  # False：没有手写的活动事务。

[('已经完成',)]
False


## 10 临时文件中的持久化与双连接观察

":memory:" 用于内存数据库；要在关闭连接后重新读取数据，可以向 connect 传入数据库文件路径。下面只使用 TemporaryDirectory 内的新文件，所有连接先关闭，再删除目录。

对于本例默认设置下的独立连接，读者看不到写者未提交的修改。读事务还可能持有快照或读锁；读完后先 rollback 结束本轮读取，再让写者提交并重新查询，避免让长时间读事务阻碍提交。

autocommit=False 的连接关闭时会回滚尚未提交的修改，不会替你提交。本例先保存 50，再把值改为 99 后直接关闭写连接；重新打开文件只应读到已经提交的 50。

In [15]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    database_path = Path(directory) / "study.sqlite3"
    with contextlib.closing(
        sqlite3.connect(database_path, autocommit=False)
    ) as writer:
        create_studies(writer)
        writer.commit()
        with contextlib.closing(
            sqlite3.connect(database_path, autocommit=False)
        ) as reader:
            with (
                contextlib.closing(writer.cursor()) as write_cursor,
                contextlib.closing(reader.cursor()) as read_cursor,
            ):
                write_cursor.execute(
                    "UPDATE studies SET minutes = ? WHERE course = ?",
                    (50, "python"),
                )
                read_cursor.execute(
                    "SELECT minutes FROM studies WHERE course = ?",
                    ("python",),
                )
                print(read_cursor.fetchall())  # [(30,)]：尚未看到写者修改。
                reader.rollback()  # 结束本轮读事务，释放读锁。

                writer.commit()
                read_cursor.execute(
                    "SELECT minutes FROM studies WHERE course = ?",
                    ("python",),
                )
                print(read_cursor.fetchall())  # [(50,)]：重新读取已提交值。
                reader.rollback()

                write_cursor.execute(
                    "UPDATE studies SET minutes = ? WHERE course = ?",
                    (99, "python"),
                )
                # 不提交 99，外层 closing 关闭 writer 时回滚此修改。

    with contextlib.closing(
        sqlite3.connect(database_path, autocommit=False)
    ) as reopened:
        with contextlib.closing(reopened.cursor()) as cursor:
            cursor.execute(
                "SELECT minutes FROM studies WHERE course = ?", ("python",)
            )
            persisted = cursor.fetchone()
            print(persisted)  # (50,)：关闭并重开后仍保留已提交值。
            assert persisted == (50,)

print(database_path.exists())  # False：全部连接关闭后，临时目录已清理。

[(30,)]
[(50,)]
(50,)
False


## 本章小结

（1）连接负责数据库会话，游标负责执行与读取。closing 安排资源关闭，with connection 安排事务提交或回滚。

（2）SQL 参数绑定传递值；列名、表名和排序方向属于 SQL 结构。参数错误与查询结果为空是不同情况。

（3）fetch 系列消费剩余结果，Row 允许按列名读取；确定结果顺序要写 ORDER BY，SELECT 的行数不能靠 rowcount 得出。

（4）Python 3.12 主要示例显式使用 autocommit=False。默认仍是旧模式；True 模式下 Python 的 commit、rollback 不起作用。

（5）同一连接看到修改不表示已经持久化；提交后重开连接读取、回滚后核对旧值，才能观察不同边界。自查：失败发生时，哪些语句属于同一事务，异常是否越过该事务的上下文边界？

## 练习

（1）先预测下面最终的课程列表，再执行核对。分别标出已提交的初始事务、with 中的事务和异常捕获位置；解释修改与新增是否保留。核对标准是预测列表的内容、数值及顺序与实际输出一致。

In [16]:
with contextlib.closing(
    sqlite3.connect(":memory:", autocommit=False)
) as exercise_connection:
    create_studies(exercise_connection)
    exercise_connection.commit()
    try:
        with exercise_connection:
            with contextlib.closing(exercise_connection.cursor()) as cursor:
                cursor.execute(
                    "UPDATE studies SET minutes = ? WHERE course = ?",
                    (5, "python"),
                )
                cursor.execute(
                    "INSERT INTO studies(course, minutes) VALUES (?, ?)",
                    ("rust", 10),
                )
            raise ValueError("本次操作取消")
    except ValueError as error:
        assert str(error) == "本次操作取消"
    else:
        raise AssertionError("演示中的取消异常应传到事务外层")

    with contextlib.closing(exercise_connection.cursor()) as cursor:
        cursor.execute("SELECT course, minutes FROM studies ORDER BY course")
        print(cursor.fetchall())
# 先记录预测；运行后按事务边界核对每条记录。

[('python', 30), ('sql', 20), ('web', 15)]


（2）编写 transfer\_minutes(connection, source, target, minutes)，在两个不同课程之间转移正整数分钟数；仅接收本章 autocommit=False 的连接，调用前没有待提交修改。用同一个事务完成扣减与增加，全部值使用参数绑定。

先拒绝非正整数或相同课程；扣减时同时要求来源余额充足，并检查两次 UPDATE 的 rowcount。来源不足、课程不存在时抛出 ValueError，使整个事务回滚。函数关闭自己的游标，连接仍由调用者关闭。

用初始表从 python 向 sql 转移 10，检查 python 为 20、sql 为 30、web 为 15。另用各自独立的初始表检查转移 100、来源不存在、目标不存在、转移 0 均失败且表内容保持原样；特别核对目标不存在时，前面的扣减也已回滚。

In [17]:
# 在此实现 transfer_minutes；函数带中文 docstring 和类型标注。
# 成功路径总分钟数保持 65；失败路径比较完整表，不能只看异常类型。
# 每次检查使用新内存连接、create_studies 和显式提交的初始数据。

（3）在 TemporaryDirectory 内创建数据库文件，用两个独立连接观察新课程 rust 的可见性。写连接插入 ("rust", 10) 后，读连接在提交前应查不到；结束读事务，让写连接提交，再开始读取时应得到一条记录。

随后写连接把 rust 改为 99 并 rollback，结束各连接事务并关闭；重新打开文件后应读到 10。显式关闭每个游标和连接，退出临时目录后检查数据库文件已不存在。不要把示例路径替换为已有数据库。

In [18]:
# 在此创建临时路径和两个 autocommit=False 的连接。
# 用 WHERE course = ? 查询，提交前结果为空，提交后为 [(10,)]。
# 读完及时结束读事务；回滚更新后重新连接核对持久值，再检查目录清理。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [sqlite3 与嵌入式 SQLite](https://docs.python.org/3.12/library/sqlite3.html#module-sqlite3)、[连接参数、内存数据库与默认事务模式](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.connect)、[创建游标](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.Connection.cursor)、[execute](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.Cursor.execute)、[executemany](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.Cursor.executemany)、[问号和命名绑定](https://docs.python.org/3.12/library/sqlite3.html#how-to-use-placeholders-to-bind-values-in-sql-queries)；[fetchone](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.Cursor.fetchone)、[fetchmany](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.Cursor.fetchmany)、[fetchall](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.Cursor.fetchall)、[rowcount](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.Cursor.rowcount)、[Row 与结果工厂](https://docs.python.org/3.12/library/sqlite3.html#how-to-create-and-use-row-factories)、[row\_factory 对新游标生效](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.Connection.row_factory)；[autocommit 三种值](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.Connection.autocommit)、[autocommit=False 的事务开启、提交和回滚](https://docs.python.org/3.12/library/sqlite3.html#transaction-control-via-the-autocommit-attribute)、[旧模式与 isolation\_level](https://docs.python.org/3.12/library/sqlite3.html#transaction-control-via-the-isolation-level-attribute)、[commit](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.Connection.commit)、[rollback](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.Connection.rollback)、[in\_transaction](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.Connection.in_transaction)、[连接上下文的提交、回滚和不关闭边界](https://docs.python.org/3.12/library/sqlite3.html#how-to-use-the-connection-context-manager)；[IntegrityError](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.IntegrityError)、[ProgrammingError](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.ProgrammingError)、[游标关闭](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.Cursor.close)、[连接关闭与未提交修改](https://docs.python.org/3.12/library/sqlite3.html#sqlite3.Connection.close)、[closing](https://docs.python.org/3.12/library/contextlib.html#contextlib.closing)、[TemporaryDirectory 清理](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory)。 |
| SQLite 官方文档 | [列定义](https://www.sqlite.org/lang_createtable.html#column_definitions)、[UNIQUE](https://www.sqlite.org/lang_createtable.html#unique_constraints)、[CHECK](https://www.sqlite.org/lang_createtable.html#check_constraints)、[NOT NULL](https://www.sqlite.org/lang_createtable.html#not_null_constraints)；[INSERT 的 VALUES 形式](https://www.sqlite.org/lang_insert.html#overview)、[WHERE 筛选](https://www.sqlite.org/lang_select.html#where_clause_filtering_)、[ORDER BY 与未指定顺序](https://www.sqlite.org/lang_select.html#the_order_by_clause)、[绑定参数表示值](https://www.sqlite.org/lang_expr.html#varparam)、[UPDATE 的 SET、WHERE 与零匹配](https://www.sqlite.org/lang_update.html#details)、[DELETE 与 WHERE](https://www.sqlite.org/lang_delete.html#overview)；[事务边界](https://www.sqlite.org/lang_transaction.html#transactions)、[DEFERRED 的实际访问时机](https://www.sqlite.org/lang_transaction.html#deferred_immediate_and_exclusive_transactions)、[自动事务完成与读锁对提交的影响](https://www.sqlite.org/lang_transaction.html#implicit_versus_explicit_transactions)、[连接间隔离及同一连接的未提交可见性](https://www.sqlite.org/isolation.html)。SQL 语法采用这些页面所述的基础形式，Python 接口行为以 3.12 文档为准。 |